In [2]:
import torch
from torch import nn

In [3]:
class Stem(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride, padding)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

    def forward(self, X):
        # X的维度是（batch, 通道，序列长度）
        X = self.bn(self.conv1(X))
        X = self.relu(X)
        return self.maxpool(X)

In [4]:
from torchvision.models.resnet import Bottleneck
class Bottleneck1D(Bottleneck):
    def __init__(self,inplanes, planes,stride=1,downsample=None):
        super().__init__(inplanes, planes, stride=stride,downsample=downsample,norm_layer=nn.BatchNorm1d)
        self.conv1 = nn.Conv1d(inplanes,planes,kernel_size=1,bias=False)
        self.conv2 = nn.Conv1d(planes,planes,kernel_size=3,stride=stride,padding=1,bias=False)
        self.conv3 = nn.Conv1d(planes,planes * self.expansion,kernel_size=1,bias=False)

class CNNBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = self._make_layer(64, 64, 3)
        self.layer2 = self._make_layer(256, 128, 4, stride=2)
        self.layer3 = self._make_layer(512, 256, 6, stride=2)
        self.layer4 = self._make_layer(1024, 512, 3, stride=2)
    def _make_layer(self,inplanes,planes,blocks,stride=1):
        downsample = None
        if stride != 1 or inplanes != planes * 4:
            downsample = nn.Sequential(
                nn.Conv1d(inplanes,planes * 4,kernel_size=1,stride=stride,bias=False),
                nn.BatchNorm1d(planes * 4)
            )

        layers = [Bottleneck1D(inplanes,planes,stride,downsample)]

        inplanes = planes * 4

        for _ in range(blocks - 1):
            layers.append(Bottleneck1D(inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

In [5]:
class TransformerBranch(nn.Module):
    def __init__(self,in_channels=64,num_layers=6,num_heads=4,mlp_ratio=4,dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=in_channels,
            nhead=num_heads,
            dim_feedforward=in_channels * mlp_ratio,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        """x:(B,C,L)"""
        x = x.transpose(1, 2) # -> (B,L,C)
        x = self.encoder(x)
        x = x.transpose(1, 2) # -> (B,C,L)
        return x

In [ ]:
class SpatialAttention(nn.Module):
    """
    主要找到单条特征的哪个位置重要，加上注意力 Input:(B, C, L);  Output: (B, C, L)
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        assert kernel_size in (3, 7), "kernel_size must be 3 or 7"
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv1d(in_channels=2,out_channels=1,kernel_size=kernel_size,padding=padding,bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        """x: (B,C,L)
        """
        avg_out = torch.mean(x, dim=1, keepdim=True) # (B,1,L)
        max_out, _ = torch.max(x, dim=1, keepdim=True) # (B,1,L)
        attention = torch.cat([avg_out, max_out], dim=1) # (B,2,L)
        attention = self.conv(attention) # (B,1,L)
        attention = self.sigmoid(attention) # (B,1,L)
        return x * attention # 广播乘法

class ChannelAttention(nn.Module):
    """
    主要找到那个通道比较重要，加上注意力  Input:(B, C, L); Output:(B, C, L)
    """
    def __init__(self, channels):
        super().__init__()

        # Global Average Pooling
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)

        # Paper: replace FC with Conv1d(kernel_size=1)
        self.conv = nn.Conv1d(in_channels=channels,out_channels=channels,kernel_size=1,bias=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # (B,C,L)
        avg_out = self.avg_pool(x)   # (B,C,1) # 每个通道只保留一个值（平均值），作者认为一个通道整体激活越强，它可能越重要。
        max_out = self.max_pool(x)   # (B,C,1) # 每个通道只保留一个值（平均值），作者认为一个通道整体激活越强，它可能越重要。
        attention = avg_out + max_out # (B,C,1)
        attention = self.conv(attention) # (B,C,1)
        attention = self.sigmoid(attention)  # (B,C,1)
        out = x * attention # broadcast
        return out
    
class FusionBlock(nn.Module):
    """
    PCTN Fusion Block: Spatial Attention -> concat -> Channel Attention
    """
    def __init__(self, channels_cnn, channels_trans):
        super().__init__()

        self.spatial_cnn = SpatialAttention()
        self.spatial_trans = SpatialAttention()
        self.channel_attention = ChannelAttention(
            channels_cnn + channels_trans
        )

    def forward(self, feat_cnn, feat_trans):
        """
        feat_cnn:     (B, C1, L)
        feat_trans:   (B, C2, L)
        """
        # 1. separate spatial attention
        cnn_feat = self.spatial_cnn(feat_cnn)
        trans_feat = self.spatial_trans(feat_trans)

        # 2. concat
        fused = torch.cat([cnn_feat, trans_feat], dim=1)  # (B, C1+C2, L)

        # 3. channel attention
        out = self.channel_attention(fused)
        return out

In [ ]:
class Regressor(nn.Module):
    """
    Regressor Module in PCTN
    Input : (B, C, L)
    Output: (B, 2) -> [SBP, DBP]
    """

    def __init__(self, in_channels,hidden_dim=128):
        super().__init__()
        # Global Average Pooling
        self.pool = nn.AdaptiveAvgPool1d(1)
        # Regression Head
        self.fc = nn.Sequential(nn.Linear(in_channels, hidden_dim),nn.ReLU(inplace=True),nn.Linear(hidden_dim, 2))

    def forward(self, x):
        """x: (B, C, L)"""
        x = self.pool(x)          # (B,C,1)
        x = x.squeeze(-1)         # (B,C)
        x = self.fc(x)            # (B,2)

        return x

In [ ]:
class PCTN(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. Stem
        self.stem = Stem(in_channels=1,out_channels=64,kernel_size=15,stride=2,padding=7)
        # 2. Two branches
        self.cnn = CNNBranch()
        self.transformer = TransformerBranch(in_channels=512, num_layers=6, num_heads=4)
        # 3. Fusion
        self.fusion = FusionBlock(channels_cnn=2048, channels_trans=512)
        # 4. Regression
        self.regressor = Regressor(in_channels=2560)

    def forward(self, x):
        """
        x: (B, 1, L) 设L为1024
        """
        # -------- Stem --------
        x = self.stem(x) # (B, 64, L') (B, 64, 256)
        # -------- CNN Branch --------
        cnn_feat = self.cnn(x)              # (B, 2048, L')
        # -------- Transformer Branch --------
        trans_feat = self.transformer(x)    # (B, 512, L')
        # ⚠️ 对齐长度（工业中必须做）
        min_len = min(cnn_feat.shape[-1], trans_feat.shape[-1])
        cnn_feat = cnn_feat[:, :, :min_len]
        trans_feat = trans_feat[:, :, :min_len]

        # -------- Fusion --------
        fused = self.fusion(cnn_feat, trans_feat)
        # -------- Regression --------
        out = self.regressor(fused)
        return out